# RBY1 × X-VLA Real Robot Inference

X-VLA FastAPI 서버(`deploy.py`)를 이용한 rby1 실로봇 inference 노트북입니다.

## 연결 구성

| 대상 | 주소 | 비고 |
|------|------|------|
| 실로봇 | `localhost:50051` | 직접 연결 |
| 시뮬레이터 | `localhost:50052` | Docker 포트 매핑 |
| X-VLA 정책 서버 | `XVLA_HOST:XVLA_PORT` | HTTP POST `/act` |

### X-VLA 서버 실행 (별도 터미널)
```bash
python deploy.py \
    --model_path 2toINF/X-VLA-Pt \
    --LoRA_path ./checkpoints/rby1_lora/ckpt-50000 \
    --action_mode auto --real_action_dim 16 \
    --port 8010
```

### 시뮬레이터 실행 (별도 터미널, 선택)
```bash
sudo docker run --rm -it \
  -e DISPLAY=$DISPLAY \
  -v /tmp/.X11-unix:/tmp/.X11-unix \
  -v /exe \
  -p 50052:50051 \
  rainbowroboticsofficial/rby1-sim
```

## 실행 순서
1. Import + Config
2. X-VLA Policy 서버 연결 확인
3. 실로봇 초기 자세 이동 (safe path + head + tool flange 12V)
4. Gripper 초기화 (homing + open)
5. RealSense 카메라 초기화 + 첫 프레임 확인
6. [선택] 시뮬레이터 초기 자세
7. [선택] 시뮬레이터 단일 Action Chunk 테스트 (X-VLA)
8. [선택] 안전 검사 + 실로봇 단일 Action Chunk (X-VLA)
9. 실로봇 Episode Loop (X-VLA)
10. 정리

## OpenPI 대비 차이점
- WebSocket `openpi_client` 대신 **HTTP POST** + `XVLARby1Client` 사용
- action shape: `[T, 20]` → `[:, :16]` trim (client 내부 처리)
- `domain_id=19` (rby1) 전달
- proprio: 16D → 20D zero-pad 후 전송 (client 내부 처리)

## Cell 1 — Import + Config

In [ ]:
from __future__ import annotations

import logging
import sys
import time

import numpy as np
import rby1_sdk as rby

# X-VLA client (lives in the X-VLA repo)
sys.path.insert(0, "..")
from evaluation.rby1.client import XVLARby1Client

logging.basicConfig(level=logging.INFO, force=True)

# ---------- User Config ----------
XVLA_HOST = "localhost"
XVLA_PORT = 8010

ROBOT_IP = "localhost:50051"
SIM_IP   = "localhost:50052"

PROMPT = "pick up the cup and place it on the plate"

# RealSense serial numbers
CAM_HEAD_SERIAL  = "922612070040"
CAM_LEFT_SERIAL  = "838212070714"
CAM_RIGHT_SERIAL = "838212074317"

# Test mode — True enables simulator / single-chunk cells
TEST_MODE = True

# Runtime / Control
MAX_HZ = 15.0
NUM_EPISODES = 1
MAX_EPISODE_STEPS = 300

ARM_COMMAND_PRIORITY = 10
ARM_MINIMUM_TIME = 10.0
LOG_ACTION_SEND = True
USE_REMOTE_GRIPPER = True

# Initial pose
ENABLE_INITIAL_POSE = True
SAFE_INIT_PATH = True
INIT_COMMAND_PRIORITY = 10
INIT_DT = 0.05
INIT_HOLD_TIME = 0.2
ENABLE_HEAD_INIT = True
HEAD_INIT_DEG = np.array([0.0, 40.0], dtype=np.float64)
HEAD_INIT = np.deg2rad(HEAD_INIT_DEG)

# Camera resolution for RealSense
IMG_WIDTH  = 640
IMG_HEIGHT = 480
RENDER_SIZE = 224

# X-VLA inference steps (action chunk length)
XVLA_STEPS = 10

print("Config loaded")
print(f"ROBOT_IP={ROBOT_IP}  SIM_IP={SIM_IP}")
print(f"XVLA server={XVLA_HOST}:{XVLA_PORT}  PROMPT={PROMPT}")

## Cell 2 — X-VLA Policy 서버 연결 확인

In [ ]:
xvla_client = XVLARby1Client(host=XVLA_HOST, port=XVLA_PORT)

ok = xvla_client.health_check()
print(f"X-VLA server health check: {'OK' if ok else 'FAILED'}")
if not ok:
    raise RuntimeError(
        f"X-VLA server at {XVLA_HOST}:{XVLA_PORT} is not responding.\n"
        "Start it with: python deploy.py --model_path 2toINF/X-VLA-Pt "
        "--LoRA_path ./checkpoints/rby1_lora/ckpt-50000 "
        "--action_mode auto --real_action_dim 16 --port 8010"
    )
print(f"XVLARby1Client ready  url={xvla_client.url}  domain_id={xvla_client.domain_id}")

## Cell 3 — 실로봇 초기 자세 이동 (safe path + head + tool flange 12V)

In [ ]:
if ENABLE_INITIAL_POSE:
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    TORSO_SLICE = slice(2, 8)
    RIGHT_SLICE = slice(8, 15)
    LEFT_SLICE = slice(15, 22)
    ELBOW_INIT_THRESHOLD_DEG = 90.0

    def _build_body_head_cmd(torso_q, right_q, left_q, head_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(robot, stream, name, torso_target, right_target, left_target, head_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)
        for i in range(steps):
            cmd = _build_body_head_cmd(torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT)
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)
        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}")
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"
    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()
    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))

    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now  = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f} deg, left={np.degrees(left_elbow_now):+.1f} deg")

    if SAFE_INIT_PATH:
        if elbow_bent:
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
        else:
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 5.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init, stream, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'OK' if ok_v else 'FAILED'}")
        time.sleep(0.5)

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

## Cell 4 — Gripper 초기화 (Homing + 열기)

In [ ]:
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("Cell 3 (initial pose) must be run first.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False -> skipped")
else:
    from examples.rby1_real.remote_gripper import Gripper as RemoteGripper

    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None

    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
        except Exception:
            pass
        gripper_obj = None

    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    ok_init = gripper_obj.initialize(verbose=True)
    if not ok_init:
        raise RuntimeError("[gripper-init] Gripper server not responding.")

    print("[gripper-init] homing...")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing failed")

    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    print(f"[gripper-init] homing done | min_q={_min_q}  max_q={_max_q}")

    gripper_obj.start()
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(1.0)

    GRIPPER_INIT_DONE = True
    print("[gripper-init] done")

## Cell 5 — RealSense 카메라 초기화 + 첫 프레임 확인

In [ ]:
import gc
import cv2
import matplotlib.pyplot as plt
import pyrealsense2 as rs

CAM_SERIALS = {
    "head":        CAM_HEAD_SERIAL,
    "left_wrist":  CAM_LEFT_SERIAL,
    "right_wrist": CAM_RIGHT_SERIAL,
}

# (A) Detect connected devices
_ctx = rs.context()
_available = {
    dev.get_info(rs.camera_info.serial_number): dev.get_info(rs.camera_info.name)
    for dev in _ctx.query_devices()
}
del _ctx

print(f"Connected RealSense devices: {len(_available)}")
for serial, name in _available.items():
    cfg_name = next((n for n, s in CAM_SERIALS.items() if s == serial), "unknown")
    mark = "OK" if serial in CAM_SERIALS.values() else "not in config"
    print(f"  [{mark}] {cfg_name:>12} | {name} | S/N: {serial}")

_not_found = [(n, s) for n, s in CAM_SERIALS.items() if s not in _available]
if _not_found:
    for n, s in _not_found:
        print(f"  MISSING: {n} ({s})")
    raise RuntimeError("Some cameras not detected. Check USB connections.")

# (B) Clean up old env
if "env" in globals() and env is not None:
    try:
        env.close()
    except Exception:
        pass
    env = None
gc.collect()
time.sleep(1.5)

# (C) Start individual camera pipelines to grab a test frame
test_frames = {}
for cam_name, serial in CAM_SERIALS.items():
    pipeline = rs.pipeline()
    cfg = rs.config()
    cfg.enable_device(serial)
    cfg.enable_stream(rs.stream.color, IMG_WIDTH, IMG_HEIGHT, rs.format.rgb8, 30)
    pipeline.start(cfg)
    for _ in range(10):
        frames = pipeline.wait_for_frames(timeout_ms=5000)
    color_frame = frames.get_color_frame()
    if color_frame:
        test_frames[cam_name] = np.asanyarray(color_frame.get_data())
    pipeline.stop()

# (D) Visualize
if test_frames:
    fig, axes = plt.subplots(1, len(test_frames), figsize=(5 * len(test_frames), 5))
    if len(test_frames) == 1:
        axes = [axes]
    for ax, (cam_name, img) in zip(axes, test_frames.items()):
        ax.imshow(img)
        ax.set_title(cam_name, fontsize=11)
        ax.axis("off")
    fig.suptitle("RealSense Test Frames", fontsize=13)
    plt.tight_layout()
    plt.show()

# Keep references for later cells
_cam_pipelines = {}
CAM_SETUP_DONE = True
print("[camera] setup done")

### Helper: capture one frame from all cameras

Shared utility used by cells 7, 8, 9.

In [ ]:
def _start_cameras():
    """Start RealSense pipelines and return dict {name: pipeline}."""
    pipelines = {}
    for cam_name, serial in CAM_SERIALS.items():
        p = rs.pipeline()
        c = rs.config()
        c.enable_device(serial)
        c.enable_stream(rs.stream.color, IMG_WIDTH, IMG_HEIGHT, rs.format.rgb8, 30)
        p.start(c)
        pipelines[cam_name] = p
    # Warm up
    for _ in range(5):
        for p in pipelines.values():
            p.wait_for_frames(timeout_ms=5000)
    return pipelines


def _stop_cameras(pipelines):
    for p in pipelines.values():
        try:
            p.stop()
        except Exception:
            pass


def _capture_images(pipelines, resize=RENDER_SIZE):
    """Capture one RGB frame per camera, resize to (resize, resize, 3) uint8."""
    images = {}
    for cam_name, p in pipelines.items():
        frames = p.wait_for_frames(timeout_ms=5000)
        color = frames.get_color_frame()
        if color:
            img = np.asanyarray(color.get_data())  # HWC RGB
            if resize is not None:
                img = cv2.resize(img, (resize, resize))
            images[cam_name] = img
    return images


def _get_robot_state_16d(robot):
    """Read robot joint state and return 16-D proprio vector.

    Layout: right_arm(7) + right_gripper(1) + left_arm(7) + left_gripper(1)
    Gripper values are set to 0.0 (unknown from joint state alone).
    """
    q = np.asarray(robot.get_state().position, dtype=np.float64)
    right_joints = q[8:15]
    left_joints  = q[15:22]
    # Gripper state is not available from robot joint state;
    # set to 0.0 (will be overridden if gripper feedback is available)
    return np.concatenate([right_joints, [0.0], left_joints, [0.0]]).astype(np.float32)


print("Helpers defined")

## Cell 6 — [선택] 시뮬레이터 초기 자세 이동

In [ ]:
if not globals().get("TEST_MODE", False):
    print("[sim-init] TEST_MODE=False -> skipped")
else:
    SIM_ADDR = SIM_IP
    SIM_INIT_DT = 0.05
    SIM_INIT_HOLD_TIME = 0.5

    TORSO_INIT_SIM = np.deg2rad(np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64))
    INIT_RIGHT_SIM = np.deg2rad(np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64))
    INIT_LEFT_SIM  = np.deg2rad(np.array([-24.0,  60.0, -10.0, -120.0,  60.0, 85.0, 0.0], dtype=np.float64))
    MID1_RIGHT_SIM = np.deg2rad(np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID1_LEFT_SIM  = np.deg2rad(np.array([70.0,  30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID2_RIGHT_SIM = np.deg2rad(np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID2_LEFT_SIM  = np.deg2rad(np.array([0.0,  15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))

    TORSO_SLICE_SIM = slice(2, 8)
    RIGHT_SLICE_SIM = slice(8, 15)
    LEFT_SLICE_SIM  = slice(15, 22)

    def _build_body_cmd_sim(torso_q, right_q, left_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        return rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        )

    def _move_waypoint_sim(robot, stream, torso_target, right_target, left_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE_SIM].copy()
        start_right = q0[RIGHT_SLICE_SIM].copy()
        start_left  = q0[LEFT_SLICE_SIM].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / SIM_INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path  = np.linspace(start_left,  left_target,  num=steps)
        for i in range(steps):
            cmd_wp = _build_body_cmd_sim(torso_path[i], right_path[i], left_path[i], dt=SIM_INIT_DT)
            try:
                stream.send_command(cmd_wp)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=10)
                    stream.send_command(cmd_wp)
                else:
                    raise
            time.sleep(SIM_INIT_DT)
        return stream

    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected(), f"Failed to connect simulator at {SIM_ADDR}"
    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-init] Failed to enable control manager")

    stream_sim = robot_sim.create_command_stream(priority=10)
    sim_waypoints = [
        (None,           MID1_RIGHT_SIM, MID1_LEFT_SIM, 5.0),
        (None,           MID2_RIGHT_SIM, MID2_LEFT_SIM, 5.0),
        (TORSO_INIT_SIM, INIT_RIGHT_SIM, INIT_LEFT_SIM, 3.0),
    ]
    for torso_t, right_t, left_t, dur_t in sim_waypoints:
        stream_sim = _move_waypoint_sim(robot_sim, stream_sim, torso_t, right_t, left_t, dur_t)

    time.sleep(0.2)
    q_init_sim = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    print("[sim-init] right:", np.round(q_init_sim[8:15], 4))
    print("[sim-init] left :", np.round(q_init_sim[15:22], 4))

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

    SIM_INITIAL_POSE_DONE = True
    print("[sim-init] done")

## Cell 7 — [선택] 시뮬레이터 단일 Action Chunk 테스트 (X-VLA)

In [ ]:
if not globals().get("TEST_MODE", False):
    print("[sim-chunk] TEST_MODE=False -> skipped")
else:
    if not globals().get("CAM_SETUP_DONE", False):
        raise RuntimeError("Run Cell 5 (camera setup) first.")
    if not globals().get("SIM_INITIAL_POSE_DONE", False):
        raise RuntimeError("Run Cell 6 (sim initial pose) first.")

    SIM_ADDR            = SIM_IP
    SIM_CHUNK_DT        = 0.1
    SIM_CHUNK_HOLD_TIME = 0.2
    SIM_CHUNK_PRIORITY  = 10

    # 1) Capture images from real cameras + sim robot state -> inference
    cam_pipes = _start_cameras()
    imgs = _capture_images(cam_pipes)

    robot_sim_chunk = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim_chunk.connect()
    assert robot_sim_chunk.is_connected()

    proprio = _get_robot_state_16d(robot_sim_chunk)

    actions = xvla_client.infer(
        head_img=imgs["head"],
        left_img=imgs["left_wrist"],
        right_img=imgs["right_wrist"],
        proprio_16d=proprio,
        instruction=PROMPT,
        steps=XVLA_STEPS,
    )
    print(f"[sim-chunk] action shape: {actions.shape}")

    right_targets = actions[:, 0:7]
    left_targets  = actions[:, 8:15]

    # 2) Send to simulator
    robot_sim_chunk.power_on(".*")
    robot_sim_chunk.servo_on(".*")
    robot_sim_chunk.reset_fault_control_manager()
    if not robot_sim_chunk.enable_control_manager():
        raise RuntimeError("[sim-chunk] Failed to enable control manager")

    q0 = np.asarray(robot_sim_chunk.get_state().position, dtype=np.float64)
    torso_hold = q0[2:8].copy()
    sim_right_start = q0[8:15].copy()
    sim_left_start  = q0[15:22].copy()

    stream_sim_chunk = robot_sim_chunk.create_command_stream(priority=SIM_CHUNK_PRIORITY)

    for i in range(actions.shape[0]):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(right_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(left_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
            )
        )
        try:
            stream_sim_chunk.send_command(cmd)
        except RuntimeError as exc:
            if "expired" in str(exc).lower():
                robot_sim_chunk.wait_for_control_ready(1000)
                stream_sim_chunk = robot_sim_chunk.create_command_stream(priority=SIM_CHUNK_PRIORITY)
                stream_sim_chunk.send_command(cmd)
            else:
                raise

        if i == 0 or (i + 1) % 5 == 0 or (i + 1) == actions.shape[0]:
            q_now = np.asarray(robot_sim_chunk.get_state().position, dtype=np.float64)
            err_r = float(np.linalg.norm(right_targets[i] - q_now[8:15]))
            err_l = float(np.linalg.norm(left_targets[i]  - q_now[15:22]))
            print(f"  step {i+1}/{actions.shape[0]} | err(r={err_r:.4f}, l={err_l:.4f})")
        time.sleep(SIM_CHUNK_DT)

    time.sleep(0.5)
    q_after = np.asarray(robot_sim_chunk.get_state().position, dtype=np.float64)
    print(f"[sim-chunk] movement norm | right={np.linalg.norm(q_after[8:15] - sim_right_start):.4f}, left={np.linalg.norm(q_after[15:22] - sim_left_start):.4f}")

    _stop_cameras(cam_pipes)
    try:
        robot_sim_chunk.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim_chunk, "disconnect"):
        robot_sim_chunk.disconnect()

## Cell 8 — [선택] 안전 검사 + 실로봇 단일 Action Chunk (X-VLA)

In [ ]:
if not globals().get("TEST_MODE", False):
    print("[safe-chunk] TEST_MODE=False -> skipped")
else:
    if not globals().get("CAM_SETUP_DONE", False):
        raise RuntimeError("Run Cell 5 first.")
    if not globals().get("INITIAL_POSE_DONE", False):
        raise RuntimeError("Run Cell 3 first.")

    SAFETY_MAX_TOTAL_NORM = 1.0
    SAFETY_MAX_STEP_DELTA = 0.15
    SAFETY_MAX_JOINT_DEG  = 30.0
    SAFETY_OVERRIDE       = True
    SAFE_DT        = 0.1
    SAFE_HOLD_TIME = 0.5
    SAFE_PRIORITY  = ARM_COMMAND_PRIORITY

    # 1) Inference
    cam_pipes = _start_cameras()
    imgs = _capture_images(cam_pipes)

    robot_safe = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_safe.connect()
    assert robot_safe.is_connected()

    proprio = _get_robot_state_16d(robot_safe)

    actions = xvla_client.infer(
        head_img=imgs["head"],
        left_img=imgs["left_wrist"],
        right_img=imgs["right_wrist"],
        proprio_16d=proprio,
        instruction=PROMPT,
        steps=XVLA_STEPS,
    )
    print(f"[safe-chunk] action shape: {actions.shape}")

    right_targets = actions[:, 0:7]
    left_targets  = actions[:, 8:15]
    right_gripper = actions[:, 7]
    left_gripper  = actions[:, 15]
    T = actions.shape[0]

    q_safe    = np.asarray(robot_safe.get_state().position, dtype=np.float64)
    torso_hold = q_safe[2:8].copy()
    cur_right  = q_safe[8:15].copy()
    cur_left   = q_safe[15:22].copy()

    # 2) Safety check
    violations = []
    total_r = float(np.linalg.norm(right_targets[-1] - right_targets[0]))
    total_l = float(np.linalg.norm(left_targets[-1]  - left_targets[0]))
    if total_r > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"right total norm {total_r:.4f} > {SAFETY_MAX_TOTAL_NORM}")
    if total_l > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"left total norm {total_l:.4f} > {SAFETY_MAX_TOTAL_NORM}")

    r_deltas = np.linalg.norm(np.diff(right_targets, axis=0), axis=1)
    l_deltas = np.linalg.norm(np.diff(left_targets,  axis=0), axis=1)
    if len(r_deltas) and r_deltas.max() > SAFETY_MAX_STEP_DELTA:
        violations.append(f"right max step delta {r_deltas.max():.4f} > {SAFETY_MAX_STEP_DELTA}")
    if len(l_deltas) and l_deltas.max() > SAFETY_MAX_STEP_DELTA:
        violations.append(f"left max step delta {l_deltas.max():.4f} > {SAFETY_MAX_STEP_DELTA}")

    disp_r = float(np.degrees(np.abs(right_targets - cur_right[None, :]).max()))
    disp_l = float(np.degrees(np.abs(left_targets  - cur_left[None,  :]).max()))
    if disp_r > SAFETY_MAX_JOINT_DEG:
        violations.append(f"right max joint displacement {disp_r:.2f} deg > {SAFETY_MAX_JOINT_DEG}")
    if disp_l > SAFETY_MAX_JOINT_DEG:
        violations.append(f"left max joint displacement {disp_l:.2f} deg > {SAFETY_MAX_JOINT_DEG}")

    if violations:
        print("Safety violations:")
        for v in violations:
            print(f"  - {v}")
        if not SAFETY_OVERRIDE:
            _stop_cameras(cam_pipes)
            robot_safe.disconnect()
            raise RuntimeError("Safety check failed. Set SAFETY_OVERRIDE=True to override.")
        print("SAFETY_OVERRIDE=True -> proceeding")
    else:
        print("[safe-chunk] Safety check passed")

    # 3) Send to real robot
    robot_safe.power_on(".*")
    robot_safe.servo_on(".*")
    robot_safe.reset_fault_control_manager()
    if not robot_safe.enable_control_manager():
        _stop_cameras(cam_pipes)
        robot_safe.disconnect()
        raise RuntimeError("[safe-chunk] Failed to enable control manager")

    stream_safe = robot_safe.create_command_stream(priority=SAFE_PRIORITY)

    for i in range(T):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(right_targets[i].astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(left_targets[i].astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
            )
        )
        try:
            stream_safe.send_command(cmd)
        except RuntimeError as exc:
            if "expired" in str(exc).lower():
                robot_safe.wait_for_control_ready(1000)
                stream_safe = robot_safe.create_command_stream(priority=SAFE_PRIORITY)
                stream_safe.send_command(cmd)
            else:
                raise

        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                gripper_obj.set_normalized_target(
                    np.array([float(right_gripper[i]), float(left_gripper[i])]),
                    wait_for_reply=False,
                )
            except Exception:
                pass

        if i == 0 or (i + 1) % 5 == 0 or (i + 1) == T:
            q_now = np.asarray(robot_safe.get_state().position, dtype=np.float64)
            err_r = float(np.linalg.norm(right_targets[i] - q_now[8:15]))
            err_l = float(np.linalg.norm(left_targets[i]  - q_now[15:22]))
            print(f"  step {i+1}/{T} | err(r={err_r:.4f}, l={err_l:.4f}) | grip(R={right_gripper[i]:+.3f}, L={left_gripper[i]:+.3f})")
        time.sleep(SAFE_DT)

    _stop_cameras(cam_pipes)

    time.sleep(0.5)
    q_final = np.asarray(robot_safe.get_state().position, dtype=np.float64)
    print(f"[safe-chunk] movement norm | right={np.linalg.norm(q_final[8:15] - cur_right):.4f}, left={np.linalg.norm(q_final[15:22] - cur_left):.4f}")

    try:
        robot_safe.cancel_control()
    except Exception:
        pass
    if hasattr(robot_safe, "disconnect"):
        robot_safe.disconnect()

## Cell 9 — 실로봇 Episode Loop (X-VLA)

In [ ]:
if not globals().get("CAM_SETUP_DONE", False):
    raise RuntimeError("Run Cell 5 first.")
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("Run Cell 3 first.")

REPLAY_DT          = 0.1
REPLAY_PRIORITY    = ARM_COMMAND_PRIORITY
REPLAY_HOLD_TIME   = 0.2
EPISODE_LENGTH     = 400
EXECUTE_CHUNK_SIZE = 8   # Execute this many steps per inference, then re-infer

CONTROL_MODE = "position"  # "position" or "impedance"

# Impedance params (only used when CONTROL_MODE == "impedance")
ARM_STIFFNESS       = np.array([80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 40.0])
ARM_DAMPING_RATIO   = 1.0
ARM_TORQUE_LIMIT    = np.array([30.0] * 7)
TORSO_STIFFNESS     = np.array([400.0] * 6)
TORSO_DAMPING_RATIO = 1.0
TORSO_TORQUE_LIMIT  = np.array([500.0] * 6)

assert CONTROL_MODE in ("position", "impedance")

def _build_arm_cmd(position, is_impedance):
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(ARM_STIFFNESS)
            .set_damping_ratio(ARM_DAMPING_RATIO)
            .set_torque_limit(ARM_TORQUE_LIMIT)
        )
    return (
        rby.JointPositionCommandBuilder()
        .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
        .set_position(position.astype(np.float64))
        .set_minimum_time(REPLAY_DT)
    )

def _build_torso_cmd(position, is_impedance):
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(TORSO_STIFFNESS)
            .set_damping_ratio(TORSO_DAMPING_RATIO)
            .set_torque_limit(TORSO_TORQUE_LIMIT)
        )
    return (
        rby.JointPositionCommandBuilder()
        .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
        .set_position(position.astype(np.float64))
        .set_minimum_time(REPLAY_DT)
    )

_use_impedance = (CONTROL_MODE == "impedance")

# Connect robot
robot = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot.connect()
assert robot.is_connected()
robot.power_on(".*")
robot.servo_on(".*")
robot.reset_fault_control_manager()
if not robot.enable_control_manager():
    robot.disconnect()
    raise RuntimeError("[episode] Failed to enable control manager")

q_init     = np.asarray(robot.get_state().position, dtype=np.float64)
ep_start_r = q_init[8:15].copy()
ep_start_l = q_init[15:22].copy()
print(f"[episode] EPISODE_LENGTH={EPISODE_LENGTH}, EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE}, CONTROL_MODE={CONTROL_MODE}")

stream      = robot.create_command_stream(priority=REPLAY_PRIORITY)
total_steps = 0
chunk_idx   = 0
send_errors = 0

# Start cameras
cam_pipes = _start_cameras()

try:
    while total_steps < EPISODE_LENGTH:
        remaining = EPISODE_LENGTH - total_steps

        # 1) Capture observation + inference
        imgs = _capture_images(cam_pipes)
        proprio = _get_robot_state_16d(robot)

        actions = xvla_client.infer(
            head_img=imgs["head"],
            left_img=imgs["left_wrist"],
            right_img=imgs["right_wrist"],
            proprio_16d=proprio,
            instruction=PROMPT,
            steps=XVLA_STEPS,
        )

        right_targets     = actions[:, 0:7]
        left_targets      = actions[:, 8:15]
        right_gripper_out = actions[:, 7]
        left_gripper_out  = actions[:, 15]

        q_now      = np.asarray(robot.get_state().position, dtype=np.float64)
        torso_hold = q_now[2:8].copy()

        chunk_size    = actions.shape[0]
        execute_size  = EXECUTE_CHUNK_SIZE if EXECUTE_CHUNK_SIZE is not None else chunk_size
        steps_to_send = min(execute_size, remaining, chunk_size)

        print(
            f"[episode] chunk {chunk_idx+1} | "
            f"steps {total_steps+1}~{total_steps+steps_to_send}/{EPISODE_LENGTH} "
            f"(execute {steps_to_send}/{chunk_size})"
        )

        # 2) Send chunk
        for i in range(steps_to_send):
            cmd = rby.RobotCommandBuilder().set_command(
                rby.ComponentBasedCommandBuilder().set_body_command(
                    rby.BodyComponentBasedCommandBuilder()
                    .set_torso_command(_build_torso_cmd(torso_hold, _use_impedance))
                    .set_right_arm_command(_build_arm_cmd(right_targets[i], _use_impedance))
                    .set_left_arm_command(_build_arm_cmd(left_targets[i], _use_impedance))
                )
            )
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                send_errors += 1
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=REPLAY_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise

            if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
                try:
                    gripper_obj.set_normalized_target(
                        np.array([float(right_gripper_out[i]), float(left_gripper_out[i])]),
                        wait_for_reply=False,
                    )
                except Exception:
                    pass

            if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
                q_diag = np.asarray(robot.get_state().position, dtype=np.float64)
                err_r = float(np.linalg.norm(right_targets[i] - q_diag[8:15]))
                err_l = float(np.linalg.norm(left_targets[i]  - q_diag[15:22]))
                moved_r = float(np.linalg.norm(q_diag[8:15]  - ep_start_r))
                moved_l = float(np.linalg.norm(q_diag[15:22] - ep_start_l))
                print(
                    f"  step {total_steps+i+1:4d}/{EPISODE_LENGTH} "
                    f"| err(r={err_r:.4f}, l={err_l:.4f}) "
                    f"| moved(r={moved_r:.4f}, l={moved_l:.4f}) "
                    f"| grip(R={right_gripper_out[i]:+.3f}, L={left_gripper_out[i]:+.3f})"
                )

            time.sleep(REPLAY_DT)

        total_steps += steps_to_send
        chunk_idx   += 1

finally:
    _stop_cameras(cam_pipes)

# Final report
time.sleep(0.5)
q_after = np.asarray(robot.get_state().position, dtype=np.float64)
print(f"\n[episode] done | steps={total_steps}, chunks={chunk_idx}, send_errors={send_errors}")
print(f"[episode] total movement | right={np.linalg.norm(q_after[8:15] - ep_start_r):.4f}, left={np.linalg.norm(q_after[15:22] - ep_start_l):.4f}")

try:
    robot.cancel_control()
except Exception:
    pass
if hasattr(robot, "disconnect"):
    robot.disconnect()

## Cell 10 — 정리 (Cleanup)

In [ ]:
# Stop gripper
if "gripper_obj" in globals() and gripper_obj is not None:
    try:
        gripper_obj.stop()
        print("[cleanup] gripper stopped")
    except Exception as e:
        print(f"[cleanup] gripper stop failed: {e}")
    gripper_obj = None

# Stop cameras
if "cam_pipes" in globals() and cam_pipes:
    _stop_cameras(cam_pipes)
    cam_pipes = {}
    print("[cleanup] cameras stopped")

# Disconnect robot
if "robot" in globals() and robot is not None:
    try:
        robot.cancel_control()
    except Exception:
        pass
    if hasattr(robot, "disconnect"):
        robot.disconnect()
    robot = None
    print("[cleanup] robot disconnected")

# Reset flags
INITIAL_POSE_DONE = False
GRIPPER_INIT_DONE = False
CAM_SETUP_DONE    = False
ENV_SETUP_DONE    = False

print("[cleanup] done")